# Step 2 — DETECTOR SELECTION · burst #21 `bn110920546`
**Status: FINALIZED** — approved by VIKAS, who supplied the reason at the gate.

ADOPT mode: the recorded human decision is *presented*, never re-adjudicated. This step
turned out to expose a divergence between the written rule and the expert's actual practice.

In [1]:
import os, json, glob, hashlib, ast, numpy as np
import astropy.io.fits as fits
from astropy.table import Table
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
BURST = "bn110920546"
sha = lambda p: hashlib.sha256(open(p,"rb").read()).hexdigest()
rel = lambda p: os.path.join(ROOT, p)
appr = json.load(open(rel(f"results/sweep106/{BURST}/APPROVALS.json")))
print("repo:", ROOT, "| burst:", BURST)

repo: /Users/salim/Desktop/Projects/SingleRest/Two_Breaks | burst: bn110920546


In [2]:
STEP = "2"

In [3]:
s = appr[STEP]
print(f'step {STEP}: {s["status"]}  by {s["by"]}  {s["utc"]}')
for f in s.get("feedback", []):
    print(f'\n  PI feedback: {f["text"][:400]}')

step 2: APPROVED  by VIKAS  2026-08-31T15:22:04Z

  PI feedback: Detector-set reason (PI, verbatim): 'I must have selected the ones those are on same side and probaly the triggered ones too' — verified: kept NaI == BCAT-triggered set exactly; all low-side so b0 in / b1 out. Written into decision.json reasoning + reasoning_provenance (marked retroactive).


## 1. The decision, and the reason (supplied retroactively at the gate)

In [4]:
d = json.load(open(rel(f"results/approval/{BURST}_decision.json")))
print("approver:", d["approver"], "| mode:", d["mode"])
print("detectors:", d["detectors"])
print("angles:", {k: round(v,2) for k,v in d["angles"].items()})
print("\nreasoning:\n ", d.get("reasoning","(none)")[:600])
print("\nprovenance of that reasoning:\n ", d.get("reasoning_provenance","")[:500])

approver: Vikas Chand | mode: human_gui
detectors: ['n0', 'n1', 'n3', 'b0']
angles: {'n0': 26.96, 'n1': 48.41, 'n3': 28.85, 'b0': 79.09}

reasoning:
  Keep the NaIs that TRIGGERED (BCAT mask) and take the BGO on the same side. Kept NaI {n0,n1,n3} == the BCAT-triggered NaI set exactly; n6 (25.33 deg) and n7 (47.68 deg) pass the <=50 deg geometry prior but are NOT in BCAT, so they were dropped despite n6 being closer than the kept n3; all kept NaIs are low-side, so the companion rule takes b0 and drops b1. The result equals the GBM science (SCAT) detector set and the set used by every published analysis of this burst (McGlynn 2012; Iyyani+2015; Li&Zhang 2021; Li 2019; Yu 2019; Wang 2024).

provenance of that reasoning:
  RETROACTIVE, not contemporaneous: the original 2026-07-19 human_gui decision recorded no reasoning field (contract gap). Supplied by the PI at the step-2 gate of the Lane-A walkthrough on 2026-08-31, VERBATIM: 'I must have selected the ones those are on same side and pro

## 2. Both halves of the reason, checked against the primitives

*"the ones those are on same side"* and *"the triggered ones"* — each is testable.

In [5]:
pend = json.load(open(rel(f"results/approval/{BURST}_pending.json")))
cands = {c["detector"]: c for c in pend.get("candidates", pend.get("detectors", []))}
kept = {x for x in d["detectors"] if x.startswith("n")}
bcat = {k for k,c in cands.items() if k.startswith("n") and c.get("in_bcat")}
LOW = set("n0 n1 n2 n3 n4 n5".split())
for k in sorted(c for c in cands if c.startswith("n")):
    c = cands[k]
    print(f'  {k}  angle {c["angle_deg"]:6.2f}  in_bcat {str(c["in_bcat"]):>5}  ->  {"KEPT" if k in kept else "dropped"}')
print(f'\n  kept NaI == triggered set exactly?  {kept == bcat}')
print(f'  all kept NaI on the low side?      {kept <= LOW}   -> companion rule takes b0, drops b1')
print(f'  pipeline pre-ticked:               {sorted(pend.get("pre_ticked", []))}')
for arm in ("claude","codex"):
    p = rel(f"results/approval/{BURST}_{arm}.json")
    if os.path.exists(p):
        a = json.load(open(p)); print(f'  {arm} AI pass kept:                {sorted(a["detectors"])}')

  n0  angle  26.96  in_bcat  True  ->  KEPT
  n1  angle  48.41  in_bcat  True  ->  KEPT
  n3  angle  28.85  in_bcat  True  ->  KEPT
  n6  angle  25.33  in_bcat False  ->  dropped
  n7  angle  47.68  in_bcat False  ->  dropped

  kept NaI == triggered set exactly?  True
  all kept NaI on the low side?      True   -> companion rule takes b0, drops b1
  pipeline pre-ticked:               ['b0', 'b1', 'n0', 'n1', 'n3', 'n6', 'n7']
  claude AI pass kept:                ['b0', 'b1', 'n0', 'n1', 'n3', 'n6', 'n7']
  codex AI pass kept:                ['b0', 'b1', 'n0', 'n1', 'n3', 'n6', 'n7']


## 3. Why this matters beyond one burst

Both AI arms followed the written rule (keep any NaI within 50°) and kept 7 detectors. The
expert kept 4. Measured across all 105 hand-approved bursts, the written rule reproduces the
expert **48%** of the time; "keep the triggered ones" reproduces it **78%**; adding side
coherence, **80%**. The ~80% ceiling is the residual light-curve judgement the skill already
requires — so the defect is not the rule but the **pre-tick prior**, which is geometry-only.
Recorded in `dev/ai_guides/detector_selection.md`; the pre-tick correction is PROPOSED, not
applied (it sits inside the benchmark's frozen approval instrument).